# Training Model Market Value Category Transfermarkt + FBref

Notebook ini menjalankan training untuk dataset gabungan Transfermarkt + FBref saja dengan dua model berbeda karakteristik: Logistic Regression dan XGBoost.

Input:
- `data/model/players_model_with_performance.csv`

Output utama:
- `models/best_model_with_performance.pkl`
- `models/label_encoder_with_performance.pkl`
- `data/output/model_metrics_with_performance.csv`
- `data/output/classification_report_best_model_with_performance.csv`
- `data/output/confusion_matrix_best_model_with_performance.csv`
- `data/output/feature_importance_best_model_with_performance.csv`


In [ ]:
import sys

!{sys.executable} -m pip install matplotlib imbalanced-learn xgboost

## 1. Import

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.modeling.train_with_performance import (
    PERFORMANCE_MODEL_DATA_FILE,
    train_dataset,
    save_results,
)

print(f'Project root: {PROJECT_ROOT}')
print(f'Model data Transfermarkt + FBref: {PERFORMANCE_MODEL_DATA_FILE}')


## 2. Train dan Evaluate

In [ ]:
result = train_dataset(PERFORMANCE_MODEL_DATA_FILE)
metrics_df = save_results(result)

print('Training Transfermarkt + FBref selesai.')
print(f'Best model: {result["best"]["model"]}')
print(f'Best scenario: {result["best"]["scenario"]}')
print(f'Best params: {result["best"]["params"]}')
print(f'Train rows: {result["train_rows"]}')
print(f'Validation rows: {result["validation_rows"]}')
print(f'Test rows: {result["test_rows"]}')


## 3. Metrics

In [ ]:
print('Top validation metrics:')
display(
    metrics_df[metrics_df['split'] == 'validation']
    .sort_values('macro_f1', ascending=False)
    .head(20)
)

print('Test metrics best model only:')
display(metrics_df[metrics_df['split'] == 'test'])


## 4. Validasi Split dan Sampling

In [ ]:
assert result['validation_rows'] > 0
assert result['test_rows'] > 0
assert result['best']['scenario'] in {'no_sampling', 'class_weight_balanced', 'hybrid_sampling_light'}
assert 'has_performance_stats' in result['features']

print('Fitur yang dipakai model:')
print(result['features'])
print()
print('Confusion matrix best model:')
print(result['test_matrix'])
print()
print('Top feature importance best model:')
display(result['feature_importance'].head(20))
print('Validasi split, skenario, fitur FBref, dan output best model lulus.')
